# Relational Graph Attention Network (RGAT) on Entities

Entity Classification on Entities (AIFB / MUTAG): Relation-specific attention mechanisms on multi-relational knowledge graphs. This notebook implements the approach with `RGATConv` inside a `K3RGATNet` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `RGATConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Relational Graph Attention Network (RGATConv) on Entities"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. RGATConv Model Definition
class K3RGATNet(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations):
        super().__init__()
        self.conv1 = k3_layers.RGATConv(in_channels, hidden_channels, num_relations=num_relations, heads=2)
        self.conv2 = k3_layers.RGATConv(hidden_channels * 2, out_channels, num_relations=num_relations, heads=1, concat=False)

    def call(self, x, edge_index, edge_type):
        x = ops.relu(self.conv1(x, edge_index, edge_type))
        return self.conv2(x, edge_index, edge_type)

k3_model = K3RGATNet(in_channels=16, hidden_channels=32, out_channels=4, num_relations=5)

# 2. Forward pass test
num_nodes = 30
dummy_x = keras.random.normal((num_nodes, 16))
dummy_edges = ops.convert_to_tensor([[0, 1, 2], [1, 2, 0]], dtype="int64")
dummy_types = ops.convert_to_tensor([0, 1, 2], dtype="int64")

out = k3_model(dummy_x, dummy_edges, dummy_types)
print(f"RGAT forward pass output shape: {out.shape}")

print("\n✓ K3-Node RGAT execution completed successfully!")